# 3_8 — Консистентность данных и устойчивость P³ (verification)

Объединяет проверки из бывших `run_consistency.py` / `run_crr_check.py` / `run_verify.py` и анализ чувствительности рейтинга P³ (`run_p3_sensitivity.py`). Post-hoc, torch не нужен.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')): REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src')); os.chdir(REPO)
import numpy as np, pandas as pd
TABLES=os.path.join(REPO,'results','tables'); ANALYSIS_TABLES=os.path.join(REPO,'results','analysis_tables')
ANALYSIS_FIGS=os.path.join(REPO,'results','analysis_figs')
for d in (ANALYSIS_TABLES,ANALYSIS_FIGS): os.makedirs(d, exist_ok=True)

## 1. Консистентность артефакта (Vs↔G0, plaxis, CRR, признаки)

In [2]:
from liquefaction_ai.data.consistency import check_artifact_consistency
ok, report = check_artifact_consistency('data/dataset')
print('\n'.join(report))
print('\nВСЕ ПРОВЕРКИ ПРОЙДЕНЫ' if ok else 'ЕСТЬ ПРОВАЛЕННЫЕ ПРОВЕРКИ')

Артефакт: 1093 образцов, 96 колонок meta
OK   risk_score_true отсутствует в meta
OK   OCR отсутствует в meta
OK   OCR отсутствует в static features
OK   PPR_max_true есть (риск-поле)
OK   Vs == sqrt(G0·1000/ρ) (digitrock), max|Δ|=0.000 м/с
OK   Vs диапазон 40–126 м/с (median 109)
OK   G0 определён для всех (median 25.0 МПа)
OK   plaxis_class совпадает на 596 образцах с грансоставом (100.0%)
OK   Cu == D60/D10, max|Δ|=0.000
OK   static_features без NaN/inf (1093, 34)
OK   CRR(N): 419/1093 образцов
plaxis_class: {'very fine': 562, 'medium': 274, 'fine': 230, 'coarse': 16, 'very coarse': 11}
DONE

ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ


## 2. Чувствительность рейтинга P³ (веса/gate/reference/risk-only)

In [3]:
from liquefaction_ai.evaluation import p3_ranking as P3
MODE='core'; NOMINAL_REF='PINN'; RISK_ONLY={'CatBoost','FT-Transformer','MLP-Risk'}; N_PERTURB=600; SEED=42
def eligible_ranking(df, reference, weights=None, gate=None):
    saved_w=dict(P3._P3_WEIGHTS[MODE]); saved_adm=P3.compute_physical_admissibility
    try:
        if weights is not None: P3._P3_WEIGHTS[MODE]=weights
        if gate is not None:
            soft,hard=gate
            P3.compute_physical_admissibility=(lambda pvr, s=soft, h=hard, k=3.0: saved_adm(pvr,s,h,k))
        table=P3.publication_ranking_table(df, reference, MODE)
    finally:
        P3._P3_WEIGHTS[MODE]=saved_w; P3.compute_physical_admissibility=saved_adm
    col='P3_Core_Admissible_Score'; okt=table[table[col].notna() & (table[col]>0)]
    return okt.sort_values(col, ascending=False)['model'].tolist()
def kendall_tau(a,b):
    common=[m for m in a if m in b]
    if len(common)<2: return float('nan')
    pa={m:i for i,m in enumerate(a)}; pb={m:i for i,m in enumerate(b)}; conc=disc=0
    for i in range(len(common)):
        for j in range(i+1,len(common)):
            x,y=common[i],common[j]; s=np.sign(pa[x]-pa[y])*np.sign(pb[x]-pb[y]); conc+=s>0; disc+=s<0
    return (conc-disc)/(conc+disc) if (conc+disc) else float('nan')

In [4]:
lb = pd.read_csv(os.path.join(TABLES,'full_leaderboard.csv')); rng=np.random.default_rng(SEED)
nominal=eligible_ranking(lb, NOMINAL_REF); top1=nominal[0]
print('Номинальный рейтинг:', nominal, '| лидер:', top1)
keys=list(P3._P3_WEIGHTS[MODE].keys()); w_nom=np.array([P3._P3_WEIGHTS[MODE][k] for k in keys]); rows=[]
taus=[]; hits=0
for _ in range(N_PERTURB):
    w=rng.dirichlet(12.0*w_nom/w_nom.sum()); r=eligible_ranking(lb, NOMINAL_REF, weights=dict(zip(keys,w)))
    taus.append(kendall_tau(nominal,r)); hits+=int(r and r[0]==top1)
taus=np.array([t for t in taus if not np.isnan(t)])
rows.append({'test':f'weights Dirichlet n={N_PERTURB}','top1_stable_%':100*hits/N_PERTURB,'kendall_tau_mean':float(taus.mean()),'kendall_tau_min':float(taus.min())})
gs=gt=0; g1=set()
for soft in (0.005,0.01,0.02):
    for hard in (0.02,0.05,0.10,0.20):
        if hard<=soft: continue
        r=eligible_ranking(lb, NOMINAL_REF, gate=(soft,hard)); gt+=1; gs+=int(r and r[0]==top1); g1.add(r[0] if r else None)
rows.append({'test':'gate thresholds grid','top1_stable_%':100*gs/gt,'kendall_tau_mean':np.nan,'kendall_tau_min':np.nan})
refs=[m for m in ('PINN','DPI-Flow','Transformer','DPI-EVT') if m in set(lb['model'])]
rr={r:eligible_ranking(lb,r) for r in refs}; ident=all(rr[r]==nominal for r in refs)
rows.append({'test':'reference model','top1_stable_%':100.0 if all(rr[r][0]==top1 for r in refs) else 0.0,'kendall_tau_mean':1.0 if ident else np.nan,'kendall_tau_min':1.0 if ident else np.nan})
lb_nr=lb[~lb['model'].isin(RISK_ONLY)].copy(); rnr=eligible_ranking(lb_nr, NOMINAL_REF)
rows.append({'test':'exclude risk-only','top1_stable_%':100.0 if (rnr and rnr[0]==top1) else 0.0,'kendall_tau_mean':kendall_tau(nominal,rnr),'kendall_tau_min':np.nan})
summary=pd.DataFrame(rows); summary.to_csv(os.path.join(ANALYSIS_TABLES,'p3_sensitivity.csv'), index=False)
summary

Номинальный рейтинг: ['EVT-NeuralSSM', 'DPI-EVT', 'Transformer', 'DPI-Flow'] | лидер: EVT-NeuralSSM


,test,top1_stable_%,kendall_tau_mean,kendall_tau_min
0,weights Dirichlet n=600,98.5,0.847778,0.0
1,gate thresholds grid,100.0,NaN,NaN
2,reference model,100.0,1.000000,1.0
3,exclude risk-only,100.0,1.000000,NaN
